# C11x · Hướng 3 — sàn theo số ứng viên (K1, K4) so với sàn cố định (A0, A8, A2), seed 4–6

Giao thức C11: YOLO26n-OBB, AI-TOD-R woX, 24 epoch, `ultralytics==8.4.174`, A100, khởi tạo `yolo26n.pt` (COCO).
Run mới ghi vào `RUNS_NEW` trên Drive, lưu checkpoint mỗi epoch (`save_period=1`), chạy lại ô train là **tự
resume**. Mọi arm (kể cả A0/A8/A2) train bằng **cùng một đường code** `c11x_train.py`.

**Cổng (preregistration §5):** chỉ chạy phần 3 khi
1. `experiments/preregistration.md` (PREREG-C11X-v1) đã có trên GitHub — ô dưới kiểm;
2. hướng 2 đã cho tín hiệu "go" theo đúng tiêu chí §5 của preregistration (ghi kết quả vào ô "GO_H2").
3. Run kiểm tái lập R0 (phần 2) khớp trong ngưỡng.

| Phần | Việc | Thời gian A100 [ước lượng: 0,55–0,57 giờ/run ở #23 của C11; chấm ~6 phút/run/head ở #29] |
|---|---|---|
| 2 | R0: train lại A2 seed 1 bằng code mới, so với run cũ | ~0,6 giờ |
| 3 | 5 arm × 3 seed = 15 run | ~8,5 giờ |
| 4 | Chấm 2 head × 15 run | ~3 giờ |
| 5 | Bảng, bootstrap B = 10.000, gộp seed | ~3 giờ |

## 0. Cấu hình

Đường dẫn giống `C11_colab.ipynb`. Code cũ (`scripts/c11`) đọc từ **Drive**; code mới (`scripts/c11x`) lấy từ
**GitHub** (`BRANCH`). Đổi `BRANCH` thành `main` sau khi PR được merge.

In [ ]:
from pathlib import Path
import sys, json, time, subprocess

DRIVE      = Path("/content/drive/MyDrive/NCKH")
CODE_DIR   = DRIVE / "repo/NCKH/scripts/c11"          # code C11 trên Drive (không có trên GitHub)
DATA_DRIVE = DRIVE / "data"
EXP_DRIVE  = DRIVE / "repo/NCKH/experiments"
LOCAL      = Path("/content/aitodr")
PROTOCOL, VERSION = "woX", "oc"
RUNS_OLD   = EXP_DRIVE / "runs-woX"                    # 12 run thăm dò của C11 (A0/A1/A8/A2 × seed 1–3)
RUNS_NEW   = EXP_DRIVE / "runs-woX-c11x"               # run mới theo preregistration
BRANCH     = "admin/c11-novelty-plan"                  # đổi thành "main" sau khi merge
GH_DIR     = Path("/content/nckh_gh")

def today():
    """Ngày lúc ghi file (không cố định từ đầu phiên như biến TODAY của C11)."""
    return time.strftime("%Y-%m-%d")

from google.colab import drive
drive.mount("/content/drive")
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "CPU runtime"

In [ ]:
!pip -q install "ultralytics==8.4.174" shapely gdown statsmodels
!rm -rf "{GH_DIR}" && git clone -q --depth 1 -b "{BRANCH}" https://github.com/VoDucNhatku/nckh-group7-tiny-obb "{GH_DIR}"
GH_COMMIT = subprocess.check_output(["git", "-C", str(GH_DIR), "rev-parse", "HEAD"]).decode().strip()
!rm -rf /content/c11 && cp -r "{CODE_DIR}" /content/c11
for p in ("/content/c11", str(GH_DIR / "scripts/c11x")):
    if p not in sys.path:
        sys.path.insert(0, p)
import ultralytics, torch, numpy, platform
ENV = dict(ultralytics=ultralytics.__version__, torch=torch.__version__, numpy=numpy.__version__,
           python=platform.python_version(), gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
           c11x_commit=GH_COMMIT)
print(ENV)
assert ENV["ultralytics"] == "8.4.174"

In [ ]:
# Cổng 1: preregistration có trên nhánh đã clone, đúng mã phiên bản
prereg = GH_DIR / "experiments" / "preregistration.md"
assert prereg.exists() and "PREREG-C11X-v1" in prereg.read_text(encoding="utf-8"), "chưa có preregistration — dừng"
print(subprocess.check_output(["git", "-C", str(GH_DIR), "log", "-1", "--format=%H %ci", "--", "experiments/preregistration.md"]).decode())

In [ ]:
# Test cũ (55, trên Drive) + test mới (bỏ file train chậm ~6 phút; chạy riêng nếu muốn)
!cd /content/c11 && python -m pytest -q test_stal_arms.py test_evalkit.py test_data_rq2.py test_bootstrap.py test_fastpath.py 2>&1 | tail -1
!cd "{GH_DIR}/scripts/c11x" && python -m pytest -q tests --ignore=tests/test_c11x_train.py -p no:cacheprovider 2>&1 | tail -1

## 1. Dữ liệu và dataset YOLO (giống hệt C11)

Giải nén nhãn + ảnh woX từ Drive, dựng lại `yolo_woX` bằng `aitodr_data.build_yolo_dataset` với **cùng `seed=0`**,
rồi so với `split_manifest.json` đã lưu ngày 20/10. Lệch thì dừng: mọi so sánh với 12 run cũ cần đúng split cũ.

In [ ]:
LOCAL.mkdir(parents=True, exist_ok=True)
lab_zip = DATA_DRIVE / "labels.zip"
assert lab_zip.exists() and open(lab_zip, "rb").read(2) == b"PK", "labels.zip chưa có trong DATA_DRIVE"
!unzip -q -o "{lab_zip}" -d "{LOCAL}"
LABELS = next(p.parent for p in LOCAL.rglob("trainval") if p.is_dir() and any(p.glob("*.txt")))
IMAGES = LOCAL / "images_raw"
IMAGES.mkdir(exist_ok=True)
for z in ["aitod_wo_xview_trainval_imgs.zip", "aitod_wo_xview_test_imgs.zip"]:
    src = DATA_DRIVE / z
    if src.exists() and not (IMAGES / (z + ".done")).exists():
        !cp "{src}" /content/ && unzip -q -o "/content/{z}" -d "{IMAGES}" && rm "/content/{z}" && touch "{IMAGES}/{z}.done"
    print(z, "OK" if (IMAGES / (z + ".done")).exists() else "CHƯA CÓ")

import aitodr_data as ad
XVIEW = ad.xview_stems()
EXCLUDE = (XVIEW["trainval"] | XVIEW["test"]) if PROTOCOL == "woX" else set()
DS = LOCAL / f"yolo_{PROTOCOL}"
man = ad.build_yolo_dataset(LABELS, IMAGES, DS, version=VERSION, exclude_xview=(PROTOCOL == "woX"),
                            val_frac=0.10, seed=0)
old = json.loads(next(EXP_DRIVE.glob(f"*-dataset-{PROTOCOL}/split_manifest.json")).read_text())
assert man["splits"] == old["splits"], ("split khác ngày 20/10", man["splits"], old["splits"])
DATA_YAML = DS / "aitodr.yaml"
print("split khớp:", man["splits"])

In [ ]:
import c11x_train as ct, c11_pipeline as cp
diff = ct.check_recipe(cp.RECIPE)
print("recipe lệch so với C11:", diff)
assert not diff, "RECIPE_C11 khác c11_pipeline.RECIPE trên Drive — sửa trước khi train"

## 2. R0 — kiểm tái lập: A2 seed 1 bằng code mới so với `c11_A2_s1` cũ

A2 trong `c11x_assign` trùng mã gốc 8.4.174 (đã test). Cùng seed, `deterministic=True`, cùng recipe nên đường
mAP50-95 trên val phải gần như trùng run cũ. Ngưỡng (preregistration §6): |ΔmAP50-95 val ở epoch 24| ≤ 0,003 và
|ΔAP test e2e| ≤ 0,002. Không đạt thì **không** dừng các so sánh mới (chúng cùng một đường code), nhưng không được
gộp run mới với 12 run cũ, và phải ghi lệch vào báo cáo.

In [ ]:
import pandas as pd
R0_DIR = EXP_DRIVE / "runs-woX-c11x-R0"
rd0 = ct.train_arm(DATA_YAML, "A2", 1, R0_DIR)
new = pd.read_csv(rd0 / "results.csv")
old = pd.read_csv(RUNS_OLD / "c11_A2_s1" / "results.csv")   # [chưa kiểm] run cũ có results.csv
col = [c for c in new.columns if "mAP50-95" in c][0]
print("mAP50-95 val epoch 24: mới", round(float(new[col].iloc[-1]), 4), "| cũ", round(float(old[col].iloc[-1]), 4))
print("lệch lớn nhất theo epoch:", float((new[col] - old[col]).abs().max()))

## 3. Train 5 arm × seed 4–6 (tự resume)

Thứ tự: seed ngoài, arm trong (các arm cùng seed chạy sát nhau, cùng loại GPU). Colab ngắt thì chạy lại ô này.

In [ ]:
GO_H2 = None   # ghi True/False theo tiêu chí go của hướng 2 (preregistration §5) TRƯỚC khi chạy ô dưới
assert GO_H2 is True, "chưa có kết quả go của hướng 2 — xem preregistration §5"
ARMS  = ["A0", "A8", "A2", "K1", "K4"]       # đúng preregistration §4; không đổi sau khi đã thấy kết quả
SEEDS = [4, 5, 6]
QUEUE = [(a, s) for s in SEEDS for a in ARMS]
for arm, seed in QUEUE:
    t0 = time.time()
    rd = ct.train_arm(DATA_YAML, arm, seed, RUNS_NEW)
    print(rd.name, "xong", f"{(time.time() - t0) / 3600:.2f} giờ phiên này")

## 4. Chấm hai head bằng AITODReval gốc (code C11 trên Drive)

In [ ]:
import importlib, aitodr_evalkit as ek, c11_bootstrap as cb
for m in (ek, cp, cb): importlib.reload(m)
TEST_IMAGES = DS / "images" / "test"
for rd in sorted(RUNS_NEW.glob("c11x_*_s*")):
    if not (rd / "c11x_done.json").exists():
        continue
    for head in ("e2e", "nms"):
        out = rd / f"eval_{head}.json"
        if out.exists():
            continue
        st = cp.predict_and_eval(rd / "weights" / "last.pt", LABELS / "test", TEST_IMAGES, out, head=head,
                                 version=VERSION, exclude_stems=EXCLUDE, save_dota_dir=rd / f"dota_{head}")
        print(rd.name, head, {k: round(st[k], 4) for k in ("AP", "AP_vt", "AP_t", "AP_s")})

## 5. Bảng, bootstrap B = 10.000, gộp bất định theo seed

Chỉ dùng run trong `RUNS_NEW` (seed 4–6). **Không** trộn với 12 run thăm dò.

In [ ]:
import pandas as pd, c11x_results as cr
from aitodr_data import scene_key
OUT = EXP_DRIVE / f"{today()}-c11x-kfloor"
OUT.mkdir(parents=True, exist_ok=True)
res = cr.collect_runs(RUNS_NEW)
display(cr.summary_table(res).round(4))
for h in ("e2e", "nms"):
    display(cr.per_seed_table(res, h).round(4))
res.to_csv(OUT / "results_long.csv", index=False)
PAIRS = [("K1", "A0"), ("K4", "A0"), ("K1", "A8"), ("K4", "A2"), ("A2", "A0"), ("A8", "A0")]
KEYS = ("AP_vt", "AP_t", "AP_s", "AP_m", "AP")
rows = []
for head in ("e2e", "nms"):
    f = OUT / f"bootstrap_{head}_B10000.json"
    boot = json.loads(f.read_text()) if f.exists() else cr.run_scene_bootstrap(cb, [RUNS_NEW], head, PAIRS, KEYS, 10_000, scene_key, f)
    for x, y in PAIRS:
        for m in KEYS:
            sc, sd = boot[f"{x}-{y}"][m], cr.paired_seed_deltas(res, head, x, y, m)
            cmb = cr.combine_scene_seed(sc, sd)
            rows.append(dict(head=head, pair=f"{x}-{y}", metric=m, delta=sc["delta"], scene_lo=sc["ci_lo"],
                             scene_hi=sc["ci_hi"], seed_min=sd.get("min"), seed_max=sd.get("max"),
                             comb_lo=cmb["lo"], comb_hi=cmb["hi"], comb_excl0=cmb.get("excludes_zero")))
ci = pd.DataFrame(rows); display(ci.round(4)); ci.to_csv(OUT / "ci_scene_seed.csv", index=False)
# Thống kê quy tắc K theo epoch: bao nhiêu GT phải nới, tới mức sàn nào
for rd in sorted(RUNS_NEW.glob("c11x_K*_s*")):
    ks = [json.loads(x) for x in (rd / "c11x_kstats.jsonl").read_text().splitlines()]
    print(rd.name, "epoch cuối:", ks[-1])
(OUT / "env.json").write_text(json.dumps(ENV, indent=2))